# Schema Design Case Studies

## Quick notes
### What this covers
- Three complete designs, each walked through the same way: plain-English requirements, entities, a text ER sketch, `CREATE TABLE` with constraints and indexes, seed data, the queries the app really needs, and design-review notes.
- Case 1: an online store (orders, order lines, price history, inventory, coupons). Table prefix `shop_`.
- Case 2: a hotel booking system (availability, overlapping-booking prevention, cancellations). Table prefix `bk_`.
- Case 3: a social and messaging app (follows, posts, likes, feeds, threads, counters, when to denormalize). Table prefix `soc_`.
- Each case ends with "what I would change at 100x scale".
- Normal forms, keys and the small library example are in 09-Design/01-Normalization-And-Design. Reporting-shaped (star) models are in 09-Design/02-Dimensional-Modeling-And-Star-Schemas. Here we design the operational (OLTP) side end to end.
### Key points
- Design from the QUERIES and the RULES backwards. List what the app must do and what must never happen, then pick tables.
- Put every rule the database can enforce into the database: `NOT NULL`, `CHECK`, `UNIQUE`, foreign keys, partial unique indexes, triggers for rules that span rows.
- Snapshot values that must not change later (the price paid on an order line, the nightly rate on a booking). Do not recompute history from today's data.
- Index for the real queries (foreign keys, feed lookups, "latest N"). See 07-Performance/01-Indexes-And-Query-Plans.
- Denormalize only on purpose (counters, feed tables) and ship a check query that detects drift.
### Gotchas (the part worth revising twice)
- Storing only "current price" on a product silently rewrites old orders. Keep price history, and copy the price onto the order line.
- A `CHECK` cannot look at other rows. "No two bookings overlap" needs a trigger (or an exclusion constraint in other databases).
- Date ranges: use half-open `[check_in, check_out)`. Then "check out on the 6th, next guest checks in on the 6th" is NOT an overlap. Two ranges overlap when `a.start < b.end AND a.end > b.start`.
- Cached counters (like_count, follower_count) drift when any write path forgets to update them. Use triggers or one write path, and keep a recount query.
- Offset pagination (`LIMIT 20 OFFSET 4000`) gets slower and skips or repeats rows when new posts arrive. Use keyset pagination on `(created_at, id)`.
- Prefixed table names (`shop_`, `bk_`, `soc_`) exist only because all three cases share one database in this notebook. In a real project they would be separate schemas or databases.
### When it matters
- Starting any new application database, or reviewing someone else's schema before it has data (cheap to change) rather than after (expensive).
- Interviews and design reviews: "walk me through the tables for X" is exactly this exercise.
### One-line recap example
- `CREATE UNIQUE INDEX one_open_price ON shop_price_history (product_id) WHERE valid_to IS NULL;` means "a product can have only one price that has no end date", the database enforces a business rule for you.

## 01. The method used in every case
- 1. **Requirements in plain English**: what must users be able to do, and what must never happen?
- 2. **Entities and relationships**: nouns become tables, "has many" becomes a foreign key, "many to many" becomes a link table.
- 3. **ER sketch** as a text diagram (the notation is explained in 09-Design/01-Normalization-And-Design).
- 4. **DDL**: types, keys, constraints, indexes. Money as integer cents, dates as ISO text.
- 5. **Seed data** small enough to check results by eye. Every query has an `ORDER BY`.
- 6. **The app queries** (5 to 8 per case). If a query is painful, go back and fix the design.
- 7. **Design review**: what is weak, what breaks at 100x scale.

## 02. Case 1, online store: requirements, entities, ER sketch
- Customers browse products and place orders. An order has one or more lines (product, quantity).
- Prices change over time. An old order must always show the price the customer actually paid.
- Stock is tracked. The shop must never sell stock it does not have (no negative stock).
- A coupon (percent off or fixed amount off) can be applied to an order. Coupons have a date window, a minimum order size and a maximum number of uses.
- Order status moves `placed -> paid -> shipped -> delivered`, or `cancelled`.
- Entities: `shop_customers`, `shop_products`, `shop_price_history`, `shop_inventory` (current stock), `shop_inventory_moves` (ledger of every change), `shop_coupons`, `shop_orders`, `shop_order_lines`.
```
shop_customers 1---<  shop_orders >---1 shop_coupons (optional)
                         |
                         1
                         |
                         ^
                   shop_order_lines >---1 shop_products 1---<  shop_price_history
                                                 |
                                                 1---1 shop_inventory
                                                 |
                                                 1---<  shop_inventory_moves
```
- Key decisions: price lives in a history table (valid_from, valid_to), not on the product. The order line copies the price. Stock changes go through a ledger table, and a trigger keeps the current number in `shop_inventory`.

## 03. Case 1: CREATE TABLE
- Money is integer cents. Dates are ISO text. Status is a `CHECK` list (see 09-Design/01-Normalization-And-Design on enums).
- `shop_inventory.on_hand >= 0` is the "never oversell" rule, enforced in the database.
- The partial unique index allows only ONE price per product with no end date (the current price).
- The trigger turns each ledger row into an update of `on_hand`. If that would go negative, the `CHECK` fails and the ledger row is rejected too (one statement, atomic).

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE shop_customers (
    customer_id INTEGER PRIMARY KEY,
    email       TEXT NOT NULL UNIQUE,
    name        TEXT NOT NULL,
    created_at  TEXT NOT NULL
);
CREATE TABLE shop_products (
    product_id INTEGER PRIMARY KEY,
    sku        TEXT NOT NULL UNIQUE,
    name       TEXT NOT NULL,
    active     INTEGER NOT NULL DEFAULT 1 CHECK (active IN (0, 1))
);
CREATE TABLE shop_price_history (
    price_id    INTEGER PRIMARY KEY,
    product_id  INTEGER NOT NULL REFERENCES shop_products,
    price_cents INTEGER NOT NULL CHECK (price_cents > 0),
    valid_from  TEXT NOT NULL,
    valid_to    TEXT,                          -- NULL = current price, [valid_from, valid_to)
    CHECK (valid_to IS NULL OR valid_to > valid_from)
);
CREATE UNIQUE INDEX ux_shop_one_open_price ON shop_price_history (product_id) WHERE valid_to IS NULL;
CREATE INDEX ix_shop_price_lookup ON shop_price_history (product_id, valid_from);

CREATE TABLE shop_inventory (
    product_id INTEGER PRIMARY KEY REFERENCES shop_products,
    on_hand    INTEGER NOT NULL DEFAULT 0 CHECK (on_hand >= 0)
);
CREATE TABLE shop_inventory_moves (
    move_id      INTEGER PRIMARY KEY,
    product_id   INTEGER NOT NULL REFERENCES shop_products,
    delta        INTEGER NOT NULL CHECK (delta <> 0),
    reason       TEXT NOT NULL CHECK (reason IN ('receipt', 'sale', 'return', 'adjustment')),
    ref_order_id INTEGER,
    created_at   TEXT NOT NULL
);
CREATE TRIGGER shop_move_updates_stock AFTER INSERT ON shop_inventory_moves
BEGIN
    UPDATE shop_inventory SET on_hand = on_hand + NEW.delta WHERE product_id = NEW.product_id;
END;

CREATE TABLE shop_coupons (
    code            TEXT PRIMARY KEY,
    kind            TEXT NOT NULL CHECK (kind IN ('percent', 'fixed')),
    value           INTEGER NOT NULL CHECK (value > 0),   -- percent (1-100) or cents
    min_order_cents INTEGER NOT NULL DEFAULT 0,
    valid_from      TEXT NOT NULL,
    valid_to        TEXT NOT NULL,
    max_uses        INTEGER,                              -- NULL = unlimited
    CHECK (kind <> 'percent' OR value <= 100)
);
CREATE TABLE shop_orders (
    order_id       INTEGER PRIMARY KEY,
    customer_id    INTEGER NOT NULL REFERENCES shop_customers,
    placed_at      TEXT NOT NULL,
    status         TEXT NOT NULL DEFAULT 'placed' CHECK (status IN ('placed', 'paid', 'shipped', 'delivered', 'cancelled')),
    coupon_code    TEXT REFERENCES shop_coupons,
    subtotal_cents INTEGER NOT NULL DEFAULT 0,
    discount_cents INTEGER NOT NULL DEFAULT 0 CHECK (discount_cents >= 0),
    shipping_cents INTEGER NOT NULL DEFAULT 0 CHECK (shipping_cents >= 0),
    total_cents    INTEGER NOT NULL DEFAULT 0
);
CREATE INDEX ix_shop_orders_customer ON shop_orders (customer_id, placed_at);
CREATE INDEX ix_shop_orders_coupon ON shop_orders (coupon_code) WHERE coupon_code IS NOT NULL;

CREATE TABLE shop_order_lines (
    order_id         INTEGER NOT NULL REFERENCES shop_orders,
    line_no          INTEGER NOT NULL,
    product_id       INTEGER NOT NULL REFERENCES shop_products,
    qty              INTEGER NOT NULL CHECK (qty > 0),
    unit_price_cents INTEGER NOT NULL CHECK (unit_price_cents > 0),   -- copied: the price paid
    PRIMARY KEY (order_id, line_no),
    UNIQUE (order_id, product_id)
) WITHOUT ROWID;
CREATE INDEX ix_shop_lines_product ON shop_order_lines (product_id);

## 04. Case 1: seed data
- Stock arrives through the ledger (a `receipt` move per product), so `shop_inventory` is filled by the trigger and always agrees with the ledger.
- Backpack (product 3) has two prices: 80000 until 2025-03-01, then a sale price of 75000. Notebook goes 5000 then 5500 from 2025-02-15.

In [3]:
%%sql
INSERT INTO shop_customers VALUES
    (1,'asha@example.com','Asha','2025-01-01'),(2,'ben@example.com','Ben','2025-01-05'),
    (3,'chen@example.com','Chen','2025-01-10'),(4,'dina@example.com','Dina','2025-02-01');
INSERT INTO shop_products (product_id, sku, name) VALUES
    (1,'P-100','Notebook'),(2,'P-200','Pen'),(3,'P-300','Backpack'),(4,'P-400','Desk Lamp'),(5,'P-500','Mug');
INSERT INTO shop_price_history (product_id, price_cents, valid_from, valid_to) VALUES
    (1, 5000,'2025-01-01','2025-02-15'),(1, 5500,'2025-02-15',NULL),
    (2, 1000,'2025-01-01',NULL),
    (3,80000,'2025-01-01','2025-03-01'),(3,75000,'2025-03-01',NULL),
    (4,60000,'2025-01-01',NULL),(5,15000,'2025-01-01',NULL);
INSERT INTO shop_coupons VALUES
    ('WELCOME10','percent',10,0,'2025-01-01','2025-12-31',2),
    ('FLAT50','fixed',5000,50000,'2025-01-01','2025-12-31',NULL),
    ('SPRING20','percent',20,0,'2025-03-01','2025-03-31',1);
INSERT INTO shop_inventory (product_id) SELECT product_id FROM shop_products;
INSERT INTO shop_inventory_moves (product_id, delta, reason, created_at) VALUES
    (1,100,'receipt','2025-01-01'),(2,500,'receipt','2025-01-01'),(3,20,'receipt','2025-01-01'),
    (4,15,'receipt','2025-01-01'),(5,50,'receipt','2025-01-01');
SELECT i.product_id, p.name, i.on_hand FROM shop_inventory i JOIN shop_products p USING (product_id) ORDER BY i.product_id;

product_id | name      | on_hand
-----------+-----------+--------
1          | Notebook  | 100    
2          | Pen       | 500    
3          | Backpack  | 20     
4          | Desk Lamp | 15     
5          | Mug       | 50     
(5 rows)



## 05. Case 1: placing orders (price as of the order date)
- Step 1: insert the order headers. Step 2: insert the lines with `INSERT ... SELECT`, looking up the price that was valid on the order date. Step 3: compute subtotal, discount and total. Step 4: write the stock ledger.
- The as-of price lookup: `valid_from <= placed_at AND (valid_to IS NULL OR placed_at < valid_to)`. Same pattern as 06-Data-Modification/04-History-Audit-And-Temporal-Data.
- The discount rule in SQL: percent coupons take a share of the subtotal, fixed coupons apply only above the minimum order, and the discount never exceeds the subtotal. Coupons only count inside their date window.
- In a real app all four steps run in ONE transaction (see 06-Data-Modification/02-Transactions). Order 5006 is cancelled, so it moves no stock.

In [4]:
%%sql
INSERT INTO shop_orders (order_id, customer_id, placed_at, status, coupon_code, shipping_cents) VALUES
    (5001,1,'2025-01-10','delivered','WELCOME10',4000),
    (5002,2,'2025-01-22','delivered',NULL,4000),
    (5003,3,'2025-02-12','delivered','FLAT50',4000),
    (5004,1,'2025-02-20','shipped',NULL,4000),
    (5005,4,'2025-03-05','paid','SPRING20',4000),
    (5006,2,'2025-03-10','cancelled',NULL,4000);

INSERT INTO shop_order_lines (order_id, line_no, product_id, qty, unit_price_cents)
WITH v(order_id, line_no, product_id, qty) AS (VALUES
    (5001,1,1,4),(5001,2,2,10),(5002,1,3,1),(5003,1,4,1),(5003,2,5,2),
    (5004,1,1,2),(5004,2,5,4),(5005,1,3,1),(5005,2,2,20),(5006,1,1,1))
SELECT v.order_id, v.line_no, v.product_id, v.qty, ph.price_cents
FROM v JOIN shop_orders o ON o.order_id = v.order_id
JOIN shop_price_history ph ON ph.product_id = v.product_id
     AND ph.valid_from <= o.placed_at AND (ph.valid_to IS NULL OR o.placed_at < ph.valid_to);

UPDATE shop_orders SET subtotal_cents = (SELECT SUM(qty * unit_price_cents) FROM shop_order_lines l WHERE l.order_id = shop_orders.order_id);
UPDATE shop_orders SET discount_cents = COALESCE((
    SELECT CASE WHEN c.kind = 'percent' THEN shop_orders.subtotal_cents * c.value / 100
                WHEN shop_orders.subtotal_cents >= c.min_order_cents THEN MIN(c.value, shop_orders.subtotal_cents)
                ELSE 0 END
    FROM shop_coupons c
    WHERE c.code = shop_orders.coupon_code AND shop_orders.placed_at BETWEEN c.valid_from AND c.valid_to), 0);
UPDATE shop_orders SET total_cents = subtotal_cents - discount_cents + shipping_cents;

INSERT INTO shop_inventory_moves (product_id, delta, reason, ref_order_id, created_at)
SELECT l.product_id, -l.qty, 'sale', l.order_id, o.placed_at
FROM shop_order_lines l JOIN shop_orders o USING (order_id)
WHERE o.status <> 'cancelled' ORDER BY l.order_id, l.line_no;

SELECT order_id, customer_id, status, coupon_code, subtotal_cents, discount_cents, shipping_cents, total_cents FROM shop_orders ORDER BY order_id;

order_id | customer_id | status    | coupon_code | subtotal_cents | discount_cents | shipping_cents | total_cents
---------+-------------+-----------+-------------+----------------+----------------+----------------+------------
5001     | 1           | delivered | WELCOME10   | 30000          | 3000           | 4000           | 31000      
5002     | 2           | delivered | NULL        | 80000          | 0              | 4000           | 84000      
5003     | 3           | delivered | FLAT50      | 90000          | 5000           | 4000           | 89000      
5004     | 1           | shipped   | NULL        | 71000          | 0              | 4000           | 75000      
5005     | 4           | paid      | SPRING20    | 95000          | 19000          | 4000           | 80000      
5006     | 2           | cancelled | NULL        | 5500           | 0              | 4000           | 9500       
(6 rows)



## 06. Case 1: the rules in action
- Each statement below breaks a rule. The database rejects it, so no application bug can create bad data. (Error text is printed instead of raised.)

In [5]:
%%sql expect_error
-- oversell: only 20 backpacks arrived, 2 were sold; selling 50 more would make stock negative
INSERT INTO shop_inventory_moves (product_id, delta, reason, created_at) VALUES (3, -50, 'sale', '2025-03-11');

ERROR: CHECK constraint failed: on_hand >= 0



In [6]:
%%sql expect_error
-- a second "current" price for Pen (partial unique index)
INSERT INTO shop_price_history (product_id, price_cents, valid_from) VALUES (2, 1200, '2025-03-01');

ERROR: UNIQUE constraint failed: shop_price_history.product_id



In [7]:
%%sql expect_error
-- quantity must be positive
INSERT INTO shop_order_lines VALUES (5006, 2, 2, 0, 1000);

ERROR: CHECK constraint failed: qty > 0



In [8]:
%%sql expect_error
-- a coupon that does not exist
INSERT INTO shop_orders (order_id, customer_id, placed_at, coupon_code) VALUES (5007, 1, '2025-03-11', 'NOPE');

ERROR: FOREIGN KEY constraint failed



In [9]:
%%sql expect_error
-- percent over 100 is nonsense
INSERT INTO shop_coupons VALUES ('BAD','percent',150,0,'2025-01-01','2025-12-31',NULL);

ERROR: CHECK constraint failed: kind <> 'percent' OR value <= 100



- The rejected oversell left no trace: the ledger row was rolled back with the failed stock update. Stock still equals the ledger total.

In [10]:
%%sql
SELECT (SELECT on_hand FROM shop_inventory WHERE product_id = 3) AS backpack_on_hand,
       (SELECT SUM(delta) FROM shop_inventory_moves WHERE product_id = 3) AS backpack_ledger_sum,
       (SELECT count(*) FROM shop_inventory_moves) AS ledger_rows;

backpack_on_hand | backpack_ledger_sum | ledger_rows
-----------------+---------------------+------------
18               | 18                  | 14         
(1 row)



## 07. Case 1: the queries the app needs
- 1. Order detail page (header and lines). 2. Price of a product as of a date. 3. Revenue by month. 4. Best sellers. 5. Low stock. 6. Coupon usage. 7. Customer value ranking. 8. Integrity checks (should return nothing).

In [11]:
%%sql
-- 1. order detail for order 5003
SELECT o.order_id, l.line_no, p.name, l.qty, l.unit_price_cents, l.qty * l.unit_price_cents AS line_total,
       o.subtotal_cents, o.discount_cents, o.shipping_cents, o.total_cents
FROM shop_orders o JOIN shop_order_lines l USING (order_id) JOIN shop_products p USING (product_id)
WHERE o.order_id = 5003 ORDER BY l.line_no;

order_id | line_no | name      | qty | unit_price_cents | line_total | subtotal_cents | discount_cents | shipping_cents | total_cents
---------+---------+-----------+-----+------------------+------------+----------------+----------------+----------------+------------
5003     | 1       | Desk Lamp | 1   | 60000            | 60000      | 90000          | 5000           | 4000           | 89000      
5003     | 2       | Mug       | 2   | 15000            | 30000      | 90000          | 5000           | 4000           | 89000      
(2 rows)



In [12]:
%%sql
-- 2. price of the Backpack and the Notebook on three dates
WITH d(day) AS (VALUES ('2025-02-01'), ('2025-02-20'), ('2025-03-05'))
SELECT d.day, p.name, ph.price_cents
FROM d CROSS JOIN shop_products p
JOIN shop_price_history ph ON ph.product_id = p.product_id
     AND ph.valid_from <= d.day AND (ph.valid_to IS NULL OR d.day < ph.valid_to)
WHERE p.name IN ('Backpack', 'Notebook') ORDER BY d.day, p.name;

day        | name     | price_cents
-----------+----------+------------
2025-02-01 | Backpack | 80000      
2025-02-01 | Notebook | 5000       
2025-02-20 | Backpack | 80000      
2025-02-20 | Notebook | 5500       
2025-03-05 | Backpack | 75000      
2025-03-05 | Notebook | 5500       
(6 rows)



In [13]:
%%sql
-- 3. revenue by month (cancelled orders excluded), count of orders, money given away in discounts
SELECT substr(placed_at, 1, 7) AS month, count(*) AS orders,
       SUM(subtotal_cents) AS gross, SUM(discount_cents) AS discounts, SUM(total_cents) AS charged
FROM shop_orders WHERE status <> 'cancelled' GROUP BY 1 ORDER BY 1;

month   | orders | gross  | discounts | charged
--------+--------+--------+-----------+--------
2025-01 | 2      | 110000 | 3000      | 115000 
2025-02 | 2      | 161000 | 5000      | 164000 
2025-03 | 1      | 95000  | 19000     | 80000  
(3 rows)



In [14]:
%%sql
-- 4. best sellers by units and revenue
SELECT p.name, SUM(l.qty) AS units, SUM(l.qty * l.unit_price_cents) AS revenue
FROM shop_order_lines l JOIN shop_orders o USING (order_id) JOIN shop_products p USING (product_id)
WHERE o.status <> 'cancelled' GROUP BY p.product_id ORDER BY revenue DESC, p.name;

name      | units | revenue
----------+-------+--------
Backpack  | 2     | 155000 
Mug       | 6     | 90000  
Desk Lamp | 1     | 60000  
Notebook  | 6     | 31000  
Pen       | 30    | 30000  
(5 rows)



In [15]:
%%sql
-- 5. low stock: reorder list (fewer than 20 left)
SELECT p.sku, p.name, i.on_hand FROM shop_inventory i JOIN shop_products p USING (product_id)
WHERE i.on_hand < 20 ORDER BY i.on_hand, p.sku;

sku   | name      | on_hand
------+-----------+--------
P-400 | Desk Lamp | 14     
P-300 | Backpack  | 18     
(2 rows)



In [16]:
%%sql
-- 6. coupon usage: uses so far vs the limit (cancelled orders do not use a coupon)
SELECT c.code, c.max_uses, count(o.order_id) AS uses,
       CASE WHEN c.max_uses IS NULL THEN 'unlimited' ELSE c.max_uses - count(o.order_id) END AS remaining
FROM shop_coupons c LEFT JOIN shop_orders o ON o.coupon_code = c.code AND o.status <> 'cancelled'
GROUP BY c.code ORDER BY c.code;

code      | max_uses | uses | remaining
----------+----------+------+----------
FLAT50    | NULL     | 1    | unlimited
SPRING20  | 1        | 1    | 0        
WELCOME10 | 2        | 1    | 1        
(3 rows)



In [17]:
%%sql
-- 7. customers ranked by money spent, with last order date
SELECT RANK() OVER (ORDER BY SUM(total_cents) DESC) AS rnk, c.name, count(*) AS orders,
       SUM(o.total_cents) AS spent, MAX(o.placed_at) AS last_order
FROM shop_customers c JOIN shop_orders o USING (customer_id)
WHERE o.status <> 'cancelled' GROUP BY c.customer_id ORDER BY rnk, c.name;

rnk | name | orders | spent  | last_order
----+------+--------+--------+-----------
1   | Asha | 2      | 106000 | 2025-02-20
2   | Chen | 1      | 89000  | 2025-02-12
3   | Ben  | 1      | 84000  | 2025-01-22
4   | Dina | 1      | 80000  | 2025-03-05
(4 rows)



In [18]:
%%sql
-- 8. integrity checks: both must return zero rows
SELECT o.order_id, o.total_cents AS stored,
       (SELECT SUM(qty * unit_price_cents) FROM shop_order_lines l WHERE l.order_id = o.order_id) - o.discount_cents + o.shipping_cents AS recomputed
FROM shop_orders o WHERE stored <> recomputed ORDER BY o.order_id;
SELECT i.product_id, i.on_hand, (SELECT SUM(delta) FROM shop_inventory_moves m WHERE m.product_id = i.product_id) AS ledger
FROM shop_inventory i WHERE i.on_hand <> ledger ORDER BY i.product_id;

order_id | stored | recomputed
---------+--------+-----------
(0 rows)

product_id | on_hand | ledger
-----------+---------+-------
(0 rows)



## 08. Case 1: design review and 100x scale
- **Good**: price history plus a price copy on the line (history safe), ledger plus trigger (auditable stock), rules in constraints, coupon limits answerable by one indexed query.
- **Weak**: stored totals can drift from the lines (hence check query 8; some teams compute them in a view instead). Coupon `max_uses` is only checked by the app: two orders at the same moment could both pass. Prices have no overlap guard (only one open price is enforced; see the trigger technique in case 2).
- **Missing on purpose**: addresses, taxes, multiple currencies, product variants (size, colour), reservations during checkout, partial shipments and returns.
- **At 100x scale I would**:
- Move to a server database with row-level locking. Decrement stock with `UPDATE ... SET on_hand = on_hand - ? WHERE product_id = ? AND on_hand >= ?` and check the affected row count, instead of read-then-write.
- Add a `reserved` quantity (or reservation rows with expiry) so carts do not oversell.
- Partition or archive `shop_orders` and `shop_order_lines` by month, and keep a small summary table (daily revenue) for dashboards. Reporting should move to a star schema: see 09-Design/02-Dimensional-Modeling-And-Star-Schemas.
- Make the ledger append-only with a periodic `on_hand` balance snapshot, so recounting does not scan years of moves.
- Enforce coupon limits with a counter row updated atomically, or a redemptions table with `UNIQUE (coupon_code, use_no)` (use_no = 1, 2, 3 ... up to `max_uses`), so two orders cannot both take the last use.
- Use an exclusion or overlap constraint for price ranges, and cache hot product prices.
- Give orders a human-facing order number and an idempotency key so a double-clicked "Pay" cannot create two orders.

## 09. Case 2, hotel booking: requirements, entities, ER sketch
- A hotel has rooms of different types (single, double, suite), each with a nightly rate.
- Guests book a room for a date range: check-in day and check-out day. A room can never be booked twice for the same night.
- A guest may check in on the day the previous guest checks out (same room, same day). That is allowed.
- A booking can be cancelled. Cancelled bookings free the room. Cancelling less than 7 days before check-in costs the first night, earlier is free.
- The rate on a booking must not change if the room type's rate changes later.
- The front desk needs: which rooms are free for a date range, occupancy per night, gaps between bookings, revenue, and today's arrivals and departures.
- The same design works for appointments (replace room by doctor or stylist, nights by time slots).
```
bk_room_types 1---< bk_rooms 1---< bk_bookings >---1 bk_guests

bk_bookings: room_id, guest_id, [check_in, check_out), status, nightly_cents (copied),
             cancelled_at, cancellation_fee_cents
rule that needs a trigger: for one room, confirmed date ranges must not overlap
```

## 10. Case 2: CREATE TABLE and the overlap triggers
- Ranges are half-open: `check_in` is the first night, `check_out` is the morning the guest leaves. A booking `[03-01, 03-04)` uses the nights of March 1, 2 and 3.
- Two ranges overlap exactly when `a.check_in < b.check_out AND a.check_out > b.check_in`. Memorise this.
- A `CHECK` cannot see other rows, so two triggers (insert and update) reject a confirmed booking that overlaps another confirmed booking of the same room. Cancelled bookings never block.
- The cancellation rule is also a `CHECK`: a booking is cancelled if and only if it has a `cancelled_at`.
- See 05-DDL-And-Integrity/02-Views-And-Triggers for trigger basics.

In [19]:
%%sql
CREATE TABLE bk_room_types (
    type_id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE,
    nightly_cents INTEGER NOT NULL CHECK (nightly_cents > 0), capacity INTEGER NOT NULL CHECK (capacity > 0)
);
CREATE TABLE bk_rooms (
    room_id INTEGER PRIMARY KEY, room_no TEXT NOT NULL UNIQUE,
    type_id INTEGER NOT NULL REFERENCES bk_room_types, floor INTEGER NOT NULL
);
CREATE TABLE bk_guests (
    guest_id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE
);
CREATE TABLE bk_bookings (
    booking_id INTEGER PRIMARY KEY,
    room_id    INTEGER NOT NULL REFERENCES bk_rooms,
    guest_id   INTEGER NOT NULL REFERENCES bk_guests,
    check_in   TEXT NOT NULL,
    check_out  TEXT NOT NULL,
    status     TEXT NOT NULL DEFAULT 'confirmed' CHECK (status IN ('confirmed', 'cancelled')),
    nightly_cents INTEGER NOT NULL CHECK (nightly_cents > 0),   -- copied from the room type at booking time
    created_at TEXT NOT NULL,
    cancelled_at TEXT,
    cancellation_fee_cents INTEGER NOT NULL DEFAULT 0 CHECK (cancellation_fee_cents >= 0),
    CHECK (check_out > check_in),
    CHECK ((status = 'cancelled') = (cancelled_at IS NOT NULL))
);
CREATE INDEX ix_bk_bookings_room_dates ON bk_bookings (room_id, check_in, check_out);
CREATE INDEX ix_bk_bookings_guest ON bk_bookings (guest_id);

CREATE TRIGGER bk_no_overlap_insert BEFORE INSERT ON bk_bookings
WHEN NEW.status = 'confirmed'
BEGIN
    SELECT RAISE(ABORT, 'room is already booked for these dates')
    WHERE EXISTS (SELECT 1 FROM bk_bookings b
                  WHERE b.room_id = NEW.room_id AND b.status = 'confirmed'
                    AND b.check_in < NEW.check_out AND b.check_out > NEW.check_in);
END;
CREATE TRIGGER bk_no_overlap_update BEFORE UPDATE OF room_id, check_in, check_out, status ON bk_bookings
WHEN NEW.status = 'confirmed'
BEGIN
    SELECT RAISE(ABORT, 'room is already booked for these dates')
    WHERE EXISTS (SELECT 1 FROM bk_bookings b
                  WHERE b.room_id = NEW.room_id AND b.status = 'confirmed' AND b.booking_id <> NEW.booking_id
                    AND b.check_in < NEW.check_out AND b.check_out > NEW.check_in);
END;

## 11. Case 2: seed data
- Five rooms, four guests, nine bookings made through the trigger (so the seed itself is checked). Booking 2 starts on the day booking 1 ends in the same room: allowed.
- The nightly rate is copied from the room type with `INSERT ... SELECT`.

In [20]:
%%sql
INSERT INTO bk_room_types VALUES (1,'Single',8000,1),(2,'Double',12000,2),(3,'Suite',25000,4);
INSERT INTO bk_rooms VALUES (1,'101',1,1),(2,'102',1,1),(3,'201',2,2),(4,'202',2,2),(5,'301',3,3);
INSERT INTO bk_guests VALUES (1,'Asha','asha@example.com'),(2,'Ben','ben@example.com'),(3,'Chen','chen@example.com'),(4,'Dina','dina@example.com');
INSERT INTO bk_bookings (booking_id, room_id, guest_id, check_in, check_out, nightly_cents, created_at)
WITH v(id, room, guest, cin, cout, made) AS (VALUES
    (1,'101',1,'2025-03-01','2025-03-04','2025-02-01'),
    (2,'101',2,'2025-03-04','2025-03-06','2025-02-05'),
    (3,'102',3,'2025-03-02','2025-03-09','2025-02-10'),
    (4,'201',1,'2025-03-03','2025-03-05','2025-02-12'),
    (5,'201',4,'2025-03-10','2025-03-12','2025-02-20'),
    (6,'301',2,'2025-03-05','2025-03-08','2025-02-22'),
    (7,'202',3,'2025-03-06','2025-03-07','2025-02-25'),
    (8,'101',4,'2025-03-08','2025-03-10','2025-02-26'),
    (9,'202',1,'2025-03-09','2025-03-11','2025-02-27'))
SELECT v.id, r.room_id, v.guest, v.cin, v.cout, t.nightly_cents, v.made
FROM v JOIN bk_rooms r ON r.room_no = v.room JOIN bk_room_types t ON t.type_id = r.type_id ORDER BY v.id;
SELECT b.booking_id, r.room_no, g.name, b.check_in, b.check_out, b.nightly_cents, b.status
FROM bk_bookings b JOIN bk_rooms r USING (room_id) JOIN bk_guests g USING (guest_id) ORDER BY b.booking_id;

booking_id | room_no | name | check_in   | check_out  | nightly_cents | status   
-----------+---------+------+------------+------------+---------------+----------
1          | 101     | Asha | 2025-03-01 | 2025-03-04 | 8000          | confirmed
2          | 101     | Ben  | 2025-03-04 | 2025-03-06 | 8000          | confirmed
3          | 102     | Chen | 2025-03-02 | 2025-03-09 | 8000          | confirmed
4          | 201     | Asha | 2025-03-03 | 2025-03-05 | 12000         | confirmed
5          | 201     | Dina | 2025-03-10 | 2025-03-12 | 12000         | confirmed
6          | 301     | Ben  | 2025-03-05 | 2025-03-08 | 25000         | confirmed
7          | 202     | Chen | 2025-03-06 | 2025-03-07 | 12000         | confirmed
8          | 101     | Dina | 2025-03-08 | 2025-03-10 | 8000          | confirmed
9          | 202     | Asha | 2025-03-09 | 2025-03-11 | 12000         | confirmed
(9 rows)



## 12. Case 2: overlap prevention and cancellations in action
- Overlap check, the canonical query: "is room X free from A to B?" is `NOT EXISTS` of any confirmed booking with `check_in < B AND check_out > A`. The trigger runs this same test for you.
- A double booking is rejected. A booking that only touches an existing one (same day out and in) is accepted.

In [21]:
%%sql expect_error
-- room 101 is taken 03-01..03-04 and 03-04..03-06: this overlaps both
INSERT INTO bk_bookings (room_id, guest_id, check_in, check_out, nightly_cents, created_at)
VALUES (1, 3, '2025-03-03', '2025-03-05', 8000, '2025-03-01');

ERROR: room is already booked for these dates



In [22]:
%%sql
-- fits exactly in the gap 03-06..03-08 (touches booking 2 and booking 8 but does not overlap)
INSERT INTO bk_bookings (room_id, guest_id, check_in, check_out, nightly_cents, created_at)
VALUES (1, 3, '2025-03-06', '2025-03-08', 8000, '2025-03-01');
SELECT booking_id, room_id, check_in, check_out FROM bk_bookings WHERE room_id = 1 ORDER BY check_in;

booking_id | room_id | check_in   | check_out 
-----------+---------+------------+-----------
1          | 1       | 2025-03-01 | 2025-03-04
2          | 1       | 2025-03-04 | 2025-03-06
10         | 1       | 2025-03-06 | 2025-03-08
8          | 1       | 2025-03-08 | 2025-03-10
(4 rows)



- Cancel booking 6 (suite, 03-05, cancelled on 03-02 which is 3 days before check-in): the fee is one night. Cancel booking 5 (03-10, cancelled on 03-02, 8 days before): no fee.
- The fee rule is one `CASE` in the `UPDATE`. `julianday` differences give whole days (see 01-Basics/05-Dates-And-Times).

In [23]:
%%sql
UPDATE bk_bookings
SET status = 'cancelled', cancelled_at = '2025-03-02',
    cancellation_fee_cents = CASE WHEN julianday(check_in) - julianday('2025-03-02') >= 7 THEN 0 ELSE nightly_cents END
WHERE booking_id IN (5, 6);
SELECT booking_id, check_in, status, cancelled_at, cancellation_fee_cents FROM bk_bookings WHERE status = 'cancelled' ORDER BY booking_id;

booking_id | check_in   | status    | cancelled_at | cancellation_fee_cents
-----------+------------+-----------+--------------+-----------------------
5          | 2025-03-10 | cancelled | 2025-03-02   | 0                     
6          | 2025-03-05 | cancelled | 2025-03-02   | 25000                 
(2 rows)



- The freed suite can be rebooked for an overlapping range. Then trying to "un-cancel" booking 6 hits the update trigger, because the room is no longer free. Also: marking a booking cancelled without `cancelled_at` breaks the `CHECK`.

In [24]:
%%sql
INSERT INTO bk_bookings (room_id, guest_id, check_in, check_out, nightly_cents, created_at)
VALUES (5, 1, '2025-03-06', '2025-03-08', 25000, '2025-03-03');
SELECT booking_id, room_id, check_in, check_out, status FROM bk_bookings WHERE room_id = 5 ORDER BY booking_id;

booking_id | room_id | check_in   | check_out  | status   
-----------+---------+------------+------------+----------
6          | 5       | 2025-03-05 | 2025-03-08 | cancelled
11         | 5       | 2025-03-06 | 2025-03-08 | confirmed
(2 rows)



In [25]:
%%sql expect_error
UPDATE bk_bookings SET status = 'confirmed', cancelled_at = NULL, cancellation_fee_cents = 0 WHERE booking_id = 6;

ERROR: room is already booked for these dates



In [26]:
%%sql expect_error
UPDATE bk_bookings SET status = 'cancelled' WHERE booking_id = 1;

ERROR: CHECK constraint failed: (status = 'cancelled') = (cancelled_at IS NOT NULL)



## 13. Case 2: the queries the front desk needs
- 1. Free rooms for a date range. 2. Occupancy per night. 3. Gaps between bookings per room. 4. Revenue by room type. 5. One guest's stays. 6. Arrivals and departures for a day. 7. Audit: any overlaps that slipped through (must be none).

In [27]:
%%sql
-- 1. rooms free for the nights of 2025-03-08 and 2025-03-09 (check-in 03-08, check-out 03-10)
SELECT t.name AS room_type, r.room_no
FROM bk_rooms r JOIN bk_room_types t USING (type_id)
WHERE NOT EXISTS (SELECT 1 FROM bk_bookings b
                  WHERE b.room_id = r.room_id AND b.status = 'confirmed'
                    AND b.check_in < '2025-03-10' AND b.check_out > '2025-03-08')
ORDER BY t.name, r.room_no;

room_type | room_no
----------+--------
Double    | 201    
Suite     | 301    
(2 rows)



In [28]:
%%sql
-- 2. occupancy per night, 03-01 to 03-10: a night is booked when check_in <= night < check_out
WITH RECURSIVE nights(night) AS (
    SELECT '2025-03-01' UNION ALL SELECT date(night, '+1 day') FROM nights WHERE night < '2025-03-10'
)
SELECT n.night, count(b.booking_id) AS rooms_occupied,
       ROUND(100.0 * count(b.booking_id) / (SELECT count(*) FROM bk_rooms), 0) AS occupancy_pct
FROM nights n LEFT JOIN bk_bookings b
     ON b.status = 'confirmed' AND b.check_in <= n.night AND n.night < b.check_out
GROUP BY n.night ORDER BY n.night;

night      | rooms_occupied | occupancy_pct
-----------+----------------+--------------
2025-03-01 | 1              | 20.0         
2025-03-02 | 2              | 40.0         
2025-03-03 | 3              | 60.0         
2025-03-04 | 3              | 60.0         
2025-03-05 | 2              | 40.0         
2025-03-06 | 4              | 80.0         
2025-03-07 | 3              | 60.0         
2025-03-08 | 2              | 40.0         
2025-03-09 | 2              | 40.0         
2025-03-10 | 1              | 20.0         
(10 rows)



In [29]:
%%sql
-- 3. gaps: free windows between consecutive bookings in the same room (LEAD = the next booking's check-in)
WITH seq AS (
    SELECT room_id, check_out AS free_from, LEAD(check_in) OVER (PARTITION BY room_id ORDER BY check_in) AS free_until
    FROM bk_bookings WHERE status = 'confirmed'
)
SELECT r.room_no, s.free_from, s.free_until, CAST(julianday(s.free_until) - julianday(s.free_from) AS INTEGER) AS free_nights
FROM seq s JOIN bk_rooms r USING (room_id)
WHERE s.free_until > s.free_from ORDER BY r.room_no, s.free_from;

room_no | free_from  | free_until | free_nights
--------+------------+------------+------------
202     | 2025-03-07 | 2025-03-09 | 2          
(1 row)



In [30]:
%%sql
-- 4. revenue by room type: nights x copied rate for confirmed bookings, plus cancellation fees
SELECT t.name AS room_type,
       SUM(CASE WHEN b.status = 'confirmed' THEN CAST(julianday(b.check_out) - julianday(b.check_in) AS INTEGER) * b.nightly_cents ELSE 0 END) AS stay_revenue,
       SUM(b.cancellation_fee_cents) AS fees,
       count(*) FILTER (WHERE b.status = 'cancelled') AS cancelled_bookings
FROM bk_bookings b JOIN bk_rooms r USING (room_id) JOIN bk_room_types t USING (type_id)
GROUP BY t.type_id ORDER BY t.name;

room_type | stay_revenue | fees  | cancelled_bookings
----------+--------------+-------+-------------------
Double    | 60000        | 0     | 1                 
Single    | 128000       | 0     | 0                 
Suite     | 50000        | 25000 | 1                 
(3 rows)



In [31]:
%%sql
-- 5. Asha's stays
SELECT b.booking_id, r.room_no, b.check_in, b.check_out,
       CAST(julianday(b.check_out) - julianday(b.check_in) AS INTEGER) AS nights,
       CAST(julianday(b.check_out) - julianday(b.check_in) AS INTEGER) * b.nightly_cents AS total_cents
FROM bk_bookings b JOIN bk_rooms r USING (room_id) WHERE b.guest_id = 1 AND b.status = 'confirmed' ORDER BY b.check_in;

booking_id | room_no | check_in   | check_out  | nights | total_cents
-----------+---------+------------+------------+--------+------------
1          | 101     | 2025-03-01 | 2025-03-04 | 3      | 24000      
4          | 201     | 2025-03-03 | 2025-03-05 | 2      | 24000      
11         | 301     | 2025-03-06 | 2025-03-08 | 2      | 50000      
9          | 202     | 2025-03-09 | 2025-03-11 | 2      | 24000      
(4 rows)



In [32]:
%%sql
-- 6. front desk on 2025-03-06: who arrives, who leaves
SELECT 'arrival' AS what, r.room_no, g.name FROM bk_bookings b JOIN bk_rooms r USING (room_id) JOIN bk_guests g USING (guest_id)
WHERE b.status = 'confirmed' AND b.check_in = '2025-03-06'
UNION ALL
SELECT 'departure', r.room_no, g.name FROM bk_bookings b JOIN bk_rooms r USING (room_id) JOIN bk_guests g USING (guest_id)
WHERE b.status = 'confirmed' AND b.check_out = '2025-03-06'
ORDER BY 1, 2;

what      | room_no | name
----------+---------+-----
arrival   | 101     | Chen
arrival   | 202     | Chen
arrival   | 301     | Asha
departure | 101     | Ben 
(4 rows)



In [33]:
%%sql
-- 7. audit: pairs of confirmed bookings that overlap in one room (expect no rows)
SELECT a.booking_id AS a, b.booking_id AS b, a.room_id
FROM bk_bookings a JOIN bk_bookings b ON a.room_id = b.room_id AND a.booking_id < b.booking_id
WHERE a.status = 'confirmed' AND b.status = 'confirmed' AND a.check_in < b.check_out AND a.check_out > b.check_in
ORDER BY a.booking_id;

a | b | room_id
--+---+--------
(0 rows)



## 14. Case 2: design review and 100x scale
- **Good**: half-open ranges, overlap rule written once and enforced by triggers, rate copied onto the booking, cancellation fee stored (so a later policy change does not alter history), audit query to prove no double bookings.
- **Weak**: the trigger runs a lookup per write. The index `(room_id, check_in, check_out)` makes it fast, but the check is not safe across concurrent connections on every database. SQLite has one writer at a time, so inside a transaction it is safe here. Dates have no time zone and no check-in time, so same-day turnover is a business rule, not a data rule.
- **Missing on purpose**: payments, deposits, room blocks for maintenance (model as a booking with a special guest or a `bk_blocks` table), overbooking policy, multiple hotels, seasonal rates.
- **At 100x scale I would**:
- Use a server database and a real exclusion constraint (PostgreSQL: `EXCLUDE USING gist (room_id WITH =, daterange(check_in, check_out) WITH &&) WHERE (status = 'confirmed')`) so concurrent requests cannot both succeed.
- Add a `hotel_id` everywhere and partition by hotel. Keep a pre-computed `bk_room_night` table (one row per room per night with a unique key `(room_id, night)`), a portable way to prevent double booking with a plain `UNIQUE` constraint, and fast availability and occupancy queries.
- Add seasonal rate plans (`rate_id`, date range, room type) and still copy the final price onto the booking.
- Soft-hold a room for 10 minutes during checkout (a `held_until` column or a separate holds table) and expire holds in a background job.
- Archive old bookings and keep an occupancy summary table for dashboards (star model: see 09-Design/02-Dimensional-Modeling-And-Star-Schemas).
- Log every status change in an audit table (see 06-Data-Modification/04-History-Audit-And-Temporal-Data).

## 15. Case 3, social and messaging app: requirements, entities, ER sketch
- Users have a unique handle. Users follow other users (one way: A follows B does not mean B follows A).
- Users write short posts (max 280 characters). A post can be a reply to another post, so conversations form threads under a root post.
- Users like posts, at most once per post.
- The home feed shows the newest top-level posts from the people a user follows, with fast paging.
- Profiles and posts show counters: followers, following, posts, likes, replies. They must be cheap to read.
- Direct messages: private threads of 2 or more members, each member has a "last read" marker so the inbox can show unread counts. A 1:1 chat between the same two people must exist only once.
- Entities: `soc_users`, `soc_follows` (link), `soc_posts` (self-referencing), `soc_likes` (link), `soc_feed_items` (precomputed feed, optional), `soc_dm_threads`, `soc_dm_members` (link), `soc_dm_messages`.
```
soc_users 1---< soc_posts >---o soc_posts (parent / root: reply threads)
   |  \             |
   |   \            1
   |    \           ^
   |     \-----< soc_likes >---1 soc_users
   |
   |---< soc_follows >---1 soc_users      (follower_id, followee_id)
   |
   |---< soc_dm_members >---1 soc_dm_threads 1---< soc_dm_messages

denormalized on purpose: follower_count, following_count, post_count (users),
                         like_count, reply_count (posts), soc_feed_items (feed rows)
```

## 16. Case 3: CREATE TABLE, indexes and counter triggers
- `soc_follows` and `soc_likes` have a composite primary key, so a duplicate follow or like is impossible, and `WITHOUT ROWID` stores them compactly.
- Extra indexes serve the reverse direction: "who follows me" `(followee_id, follower_id)` and "who liked this post" `(post_id)`.
- The feed index `(author_id, created_at DESC, post_id DESC)` serves "latest posts by this author", which a feed query repeats for every followed user.
- Counters are columns on `soc_users` and `soc_posts`. Triggers keep them in step with the link tables, so a follow, unfollow, like, unlike or new post updates them. Deleting a post is not covered: see the review in section 22. This is denormalization on purpose: reading a profile is one row, not three `COUNT(*)` queries.
- `soc_dm_threads.pair_key` (like `'1:2'`, smaller id first) is `UNIQUE`, so a 1:1 chat can exist once. Group threads leave it NULL (many NULLs are allowed in a `UNIQUE` column).

In [34]:
%%sql
CREATE TABLE soc_users (
    user_id INTEGER PRIMARY KEY, handle TEXT NOT NULL UNIQUE, display_name TEXT NOT NULL, created_at TEXT NOT NULL,
    follower_count  INTEGER NOT NULL DEFAULT 0,
    following_count INTEGER NOT NULL DEFAULT 0,
    post_count      INTEGER NOT NULL DEFAULT 0
);
CREATE TABLE soc_follows (
    follower_id INTEGER NOT NULL REFERENCES soc_users, followee_id INTEGER NOT NULL REFERENCES soc_users,
    created_at TEXT NOT NULL,
    PRIMARY KEY (follower_id, followee_id),
    CHECK (follower_id <> followee_id)
) WITHOUT ROWID;
CREATE INDEX ix_soc_followers ON soc_follows (followee_id, follower_id);

CREATE TABLE soc_posts (
    post_id INTEGER PRIMARY KEY,
    author_id INTEGER NOT NULL REFERENCES soc_users,
    body TEXT NOT NULL CHECK (length(body) BETWEEN 1 AND 280),
    created_at TEXT NOT NULL,
    parent_post_id INTEGER REFERENCES soc_posts,       -- direct reply target
    root_post_id   INTEGER REFERENCES soc_posts,       -- top of the thread (NULL for a top-level post)
    like_count  INTEGER NOT NULL DEFAULT 0,
    reply_count INTEGER NOT NULL DEFAULT 0,
    CHECK ((parent_post_id IS NULL) = (root_post_id IS NULL))
);
CREATE INDEX ix_soc_posts_author_time ON soc_posts (author_id, created_at DESC, post_id DESC);
CREATE INDEX ix_soc_posts_thread ON soc_posts (root_post_id, created_at) WHERE root_post_id IS NOT NULL;

CREATE TABLE soc_likes (
    user_id INTEGER NOT NULL REFERENCES soc_users, post_id INTEGER NOT NULL REFERENCES soc_posts,
    created_at TEXT NOT NULL, PRIMARY KEY (user_id, post_id)
) WITHOUT ROWID;
CREATE INDEX ix_soc_likes_post ON soc_likes (post_id);

CREATE TABLE soc_feed_items (
    user_id INTEGER NOT NULL REFERENCES soc_users, post_id INTEGER NOT NULL REFERENCES soc_posts,
    created_at TEXT NOT NULL, PRIMARY KEY (user_id, created_at DESC, post_id DESC)
) WITHOUT ROWID;

CREATE TABLE soc_dm_threads (
    thread_id INTEGER PRIMARY KEY, pair_key TEXT UNIQUE, created_at TEXT NOT NULL
);
CREATE TABLE soc_dm_members (
    thread_id INTEGER NOT NULL REFERENCES soc_dm_threads, user_id INTEGER NOT NULL REFERENCES soc_users,
    last_read_message_id INTEGER NOT NULL DEFAULT 0, PRIMARY KEY (thread_id, user_id)
) WITHOUT ROWID;
CREATE INDEX ix_soc_dm_members_user ON soc_dm_members (user_id);
CREATE TABLE soc_dm_messages (
    message_id INTEGER PRIMARY KEY, thread_id INTEGER NOT NULL REFERENCES soc_dm_threads,
    sender_id INTEGER NOT NULL REFERENCES soc_users, body TEXT NOT NULL, sent_at TEXT NOT NULL
);
CREATE INDEX ix_soc_dm_messages_thread ON soc_dm_messages (thread_id, message_id);

-- counter triggers
CREATE TRIGGER soc_follow_add AFTER INSERT ON soc_follows BEGIN
    UPDATE soc_users SET following_count = following_count + 1 WHERE user_id = NEW.follower_id;
    UPDATE soc_users SET follower_count  = follower_count + 1  WHERE user_id = NEW.followee_id;
END;
CREATE TRIGGER soc_follow_del AFTER DELETE ON soc_follows BEGIN
    UPDATE soc_users SET following_count = following_count - 1 WHERE user_id = OLD.follower_id;
    UPDATE soc_users SET follower_count  = follower_count - 1  WHERE user_id = OLD.followee_id;
END;
CREATE TRIGGER soc_like_add AFTER INSERT ON soc_likes BEGIN
    UPDATE soc_posts SET like_count = like_count + 1 WHERE post_id = NEW.post_id;
END;
CREATE TRIGGER soc_like_del AFTER DELETE ON soc_likes BEGIN
    UPDATE soc_posts SET like_count = like_count - 1 WHERE post_id = OLD.post_id;
END;
CREATE TRIGGER soc_post_add AFTER INSERT ON soc_posts BEGIN
    UPDATE soc_users SET post_count = post_count + 1 WHERE user_id = NEW.author_id;
    UPDATE soc_posts SET reply_count = reply_count + 1 WHERE post_id = NEW.parent_post_id;
END;

## 17. Case 3: seed data
- Five users, a small follow graph, eleven posts (two threads), a few likes, and three direct-message threads. Counters start at 0 and the triggers fill them in.
- Posts 4, 5 and 7 reply inside thread 1. Post 9 replies to post 8.

In [35]:
%%sql
INSERT INTO soc_users (user_id, handle, display_name, created_at) VALUES
    (1,'ana','Ana','2025-01-01'),(2,'bo','Bo','2025-01-02'),(3,'cy','Cy','2025-01-03'),
    (4,'di','Di','2025-01-04'),(5,'eli','Eli','2025-01-05');
INSERT INTO soc_follows VALUES
    (1,2,'2025-02-01'),(1,3,'2025-02-01'),(2,1,'2025-02-02'),(2,3,'2025-02-03'),
    (3,4,'2025-02-04'),(4,1,'2025-02-05'),(5,1,'2025-02-06');
INSERT INTO soc_posts (post_id, author_id, body, created_at, parent_post_id, root_post_id) VALUES
    (1,1,'Hello world','2025-03-01 09:00',NULL,NULL),
    (2,2,'SQL tip: use keyset pagination','2025-03-01 10:30',NULL,NULL),
    (3,3,'Anyone up for lunch?','2025-03-02 08:15',NULL,NULL),
    (4,2,'Welcome!','2025-03-02 09:00',1,1),
    (5,1,'Thanks Bo','2025-03-02 09:30',4,1),
    (6,4,'New blog post is up','2025-03-03 12:00',NULL,NULL),
    (7,3,'Hi Ana','2025-03-03 13:00',1,1),
    (8,1,'Weekend plans?','2025-03-04 18:00',NULL,NULL),
    (9,2,'Hiking','2025-03-04 18:30',8,8),
    (10,3,'Lunch was great','2025-03-05 08:00',NULL,NULL),
    (11,2,'Reading about indexes','2025-03-05 09:00',NULL,NULL);
INSERT INTO soc_likes VALUES
    (2,1,'2025-03-01 09:10'),(3,1,'2025-03-01 09:20'),(4,1,'2025-03-01 11:00'),
    (1,2,'2025-03-01 10:40'),(3,2,'2025-03-01 12:00'),(1,6,'2025-03-03 12:30'),
    (2,6,'2025-03-03 14:00'),(5,8,'2025-03-04 19:00'),(1,10,'2025-03-05 08:30');
INSERT INTO soc_dm_threads VALUES (1,'1:2','2025-03-04 10:00'),(2,NULL,'2025-03-04 11:00'),(3,'2:3','2025-03-05 09:00');
INSERT INTO soc_dm_members VALUES (1,1,3),(1,2,4),(2,1,5),(2,3,0),(2,4,7),(3,2,8),(3,3,9);
INSERT INTO soc_dm_messages (message_id, thread_id, sender_id, body, sent_at) VALUES
    (1,1,1,'Lunch?','2025-03-04 10:00'),(2,1,2,'Sure','2025-03-04 10:05'),
    (3,1,1,'Noon?','2025-03-04 10:06'),(4,1,2,'See you','2025-03-04 10:10'),
    (5,2,3,'Group plan?','2025-03-04 11:00'),(6,2,4,'I am in','2025-03-04 11:05'),(7,2,1,'Me too','2025-03-04 11:10'),
    (8,3,2,'Got the report?','2025-03-05 09:00'),(9,3,3,'Sending now','2025-03-05 09:30');
SELECT user_id, handle, follower_count, following_count, post_count FROM soc_users ORDER BY user_id;
SELECT post_id, author_id, like_count, reply_count FROM soc_posts ORDER BY post_id;

user_id | handle | follower_count | following_count | post_count
--------+--------+----------------+-----------------+-----------
1       | ana    | 3              | 2               | 3         
2       | bo     | 1              | 2               | 4         
3       | cy     | 2              | 1               | 3         
4       | di     | 1              | 1               | 1         
5       | eli    | 0              | 1               | 0         
(5 rows)

post_id | author_id | like_count | reply_count
--------+-----------+------------+------------
1       | 1         | 3          | 2          
2       | 2         | 2          | 0          
3       | 3         | 0          | 0          
4       | 2         | 0          | 1          
5       | 1         | 0          | 0          
6       | 4         | 2          | 0          
7       | 3         | 0          | 0          
8       | 1         | 1          | 1          
9       | 2         | 0          | 0          
10      | 3       

## 18. Case 3: counters, rules and drift
- Duplicate follow, self follow, duplicate like and a duplicate 1:1 chat are all rejected by keys and constraints.

In [36]:
%%sql expect_error
INSERT INTO soc_follows VALUES (1, 2, '2025-03-01');

ERROR: UNIQUE constraint failed: soc_follows.follower_id, soc_follows.followee_id



In [37]:
%%sql expect_error
INSERT INTO soc_follows VALUES (3, 3, '2025-03-01');

ERROR: CHECK constraint failed: follower_id <> followee_id



In [38]:
%%sql expect_error
INSERT INTO soc_likes VALUES (2, 1, '2025-03-06 10:00');

ERROR: UNIQUE constraint failed: soc_likes.user_id, soc_likes.post_id



In [39]:
%%sql expect_error
INSERT INTO soc_dm_threads (pair_key, created_at) VALUES ('1:2', '2025-03-06 10:00');

ERROR: UNIQUE constraint failed: soc_dm_threads.pair_key



- Unfollow and unlike go through the triggers too. Eli unfollows Ana and Ana un-likes post 10, the counters follow.

In [40]:
%%sql
DELETE FROM soc_follows WHERE follower_id = 5 AND followee_id = 1;
DELETE FROM soc_likes WHERE user_id = 1 AND post_id = 10;
SELECT user_id, follower_count FROM soc_users WHERE user_id = 1;
SELECT post_id, like_count FROM soc_posts WHERE post_id = 10;

user_id | follower_count
--------+---------------
1       | 2             
(1 row)

post_id | like_count
--------+-----------
10      | 0         
(1 row)



- **Drift**: a counter is a copy of a fact that can be recomputed. If a bulk import, a manual fix or a bug bypasses the triggers, the copy goes wrong. Always ship a recount query that finds mismatches, and a repair statement.
- Below: someone "fixes" a number by hand (Di shows 99 followers), then the recount query finds it.

In [41]:
%%sql
UPDATE soc_users SET follower_count = 99 WHERE user_id = 4;
SELECT u.user_id, u.handle, u.follower_count AS stored,
       (SELECT count(*) FROM soc_follows f WHERE f.followee_id = u.user_id) AS actual
FROM soc_users u
WHERE u.follower_count <> (SELECT count(*) FROM soc_follows f WHERE f.followee_id = u.user_id)
   OR u.following_count <> (SELECT count(*) FROM soc_follows f WHERE f.follower_id = u.user_id)
   OR u.post_count <> (SELECT count(*) FROM soc_posts p WHERE p.author_id = u.user_id)
ORDER BY u.user_id;

user_id | handle | stored | actual
--------+--------+--------+-------
4       | di     | 99     | 1     
(1 row)



In [42]:
%%sql
-- repair: recompute all counters from the source tables (safe to run any time)
UPDATE soc_users SET
    follower_count  = (SELECT count(*) FROM soc_follows f WHERE f.followee_id = soc_users.user_id),
    following_count = (SELECT count(*) FROM soc_follows f WHERE f.follower_id = soc_users.user_id),
    post_count      = (SELECT count(*) FROM soc_posts p WHERE p.author_id = soc_users.user_id);
UPDATE soc_posts SET
    like_count  = (SELECT count(*) FROM soc_likes l WHERE l.post_id = soc_posts.post_id),
    reply_count = (SELECT count(*) FROM soc_posts r WHERE r.parent_post_id = soc_posts.post_id);
SELECT user_id, handle, follower_count, following_count, post_count FROM soc_users ORDER BY user_id;

user_id | handle | follower_count | following_count | post_count
--------+--------+----------------+-----------------+-----------
1       | ana    | 2              | 2               | 3         
2       | bo     | 1              | 2               | 4         
3       | cy     | 2              | 1               | 3         
4       | di     | 1              | 1               | 1         
5       | eli    | 0              | 0               | 0         
(5 rows)



## 19. Case 3: the home feed, two ways
- **Fan-out on read** (pull): at read time, find who the user follows and take their latest posts. Nothing extra to store. Cost grows with the number of people followed.
- Use **keyset pagination**: remember the `(created_at, post_id)` of the last row you showed and ask for rows strictly older. It is stable when new posts arrive and does not slow down on deep pages, unlike `OFFSET`.
- Only top-level posts go in the feed (`parent_post_id IS NULL`).
- Page 1 (2 posts) for Ana, then page 2 using the last row of page 1.

In [43]:
%%sql
-- page 1
SELECT p.post_id, u.handle, p.body, p.created_at
FROM soc_follows f JOIN soc_posts p ON p.author_id = f.followee_id
JOIN soc_users u ON u.user_id = p.author_id
WHERE f.follower_id = 1 AND p.parent_post_id IS NULL
ORDER BY p.created_at DESC, p.post_id DESC LIMIT 2;
-- page 2: strictly older than the last row of page 1 (2025-03-05 08:00, post 10)
SELECT p.post_id, u.handle, p.body, p.created_at
FROM soc_follows f JOIN soc_posts p ON p.author_id = f.followee_id
JOIN soc_users u ON u.user_id = p.author_id
WHERE f.follower_id = 1 AND p.parent_post_id IS NULL
  AND (p.created_at, p.post_id) < ('2025-03-05 08:00', 10)
ORDER BY p.created_at DESC, p.post_id DESC LIMIT 2;

post_id | handle | body                  | created_at      
--------+--------+-----------------------+-----------------
11      | bo     | Reading about indexes | 2025-03-05 09:00
10      | cy     | Lunch was great       | 2025-03-05 08:00
(2 rows)

post_id | handle | body                           | created_at      
--------+--------+--------------------------------+-----------------
3       | cy     | Anyone up for lunch?           | 2025-03-02 08:15
2       | bo     | SQL tip: use keyset pagination | 2025-03-01 10:30
(2 rows)



- **Fan-out on write** (push): when a post is created, copy a row into each follower's feed table (`soc_feed_items`). Reading a feed is then one indexed range scan. Cost moves to write time: a user with a million followers writes a million rows.
- Real systems mix them: push for normal users, pull for celebrities, merged at read time.
- Below we fill the feed table for all existing top-level posts with one `INSERT ... SELECT`, then read Ana's feed from it. The result matches the pull query.

In [44]:
%%sql
INSERT INTO soc_feed_items (user_id, post_id, created_at)
SELECT f.follower_id, p.post_id, p.created_at
FROM soc_posts p JOIN soc_follows f ON f.followee_id = p.author_id
WHERE p.parent_post_id IS NULL;
SELECT fi.post_id, u.handle, p.body, fi.created_at
FROM soc_feed_items fi JOIN soc_posts p USING (post_id) JOIN soc_users u ON u.user_id = p.author_id
WHERE fi.user_id = 1 ORDER BY fi.created_at DESC, fi.post_id DESC;

post_id | handle | body                           | created_at      
--------+--------+--------------------------------+-----------------
11      | bo     | Reading about indexes          | 2025-03-05 09:00
10      | cy     | Lunch was great                | 2025-03-05 08:00
3       | cy     | Anyone up for lunch?           | 2025-03-02 08:15
2       | bo     | SQL tip: use keyset pagination | 2025-03-01 10:30
(4 rows)



- Cost of the push table: it must be kept right when someone unfollows, a post is deleted, or the privacy rules change. That is why it is the last thing to add, not the first.

## 20. Case 3: threads and "who to follow"
- **Threads**: `root_post_id` lets us load a whole conversation with one indexed lookup, already in time order. A recursive CTE on `parent_post_id` gives nesting depth for indentation (see 02-Querying/03-Subqueries-And-CTEs and 04-Patterns/06-Hierarchies-And-Graphs).

In [45]:
%%sql
-- flat view of thread 1, in time order (uses root_post_id)
SELECT p.post_id, u.handle, p.body, p.parent_post_id
FROM soc_posts p JOIN soc_users u ON u.user_id = p.author_id
WHERE p.post_id = 1 OR p.root_post_id = 1 ORDER BY p.created_at, p.post_id;
-- nested view with depth (recursive, walks parent links)
WITH RECURSIVE t(post_id, depth, path) AS (
    SELECT post_id, 0, printf('%05d', post_id) FROM soc_posts WHERE post_id = 1
    UNION ALL
    SELECT c.post_id, t.depth + 1, t.path || '/' || printf('%05d', c.post_id)
    FROM soc_posts c JOIN t ON c.parent_post_id = t.post_id
)
SELECT t.depth, substr('        ', 1, t.depth * 2) || u.handle || ': ' || p.body AS line
FROM t JOIN soc_posts p USING (post_id) JOIN soc_users u ON u.user_id = p.author_id ORDER BY t.path;

post_id | handle | body        | parent_post_id
--------+--------+-------------+---------------
1       | ana    | Hello world | NULL          
4       | bo     | Welcome!    | 1             
5       | ana    | Thanks Bo   | 4             
7       | cy     | Hi Ana      | 1             
(4 rows)

depth | line              
------+-------------------
0     | ana: Hello world  
1     |   bo: Welcome!    
2     |     ana: Thanks Bo
1     |   cy: Hi Ana      
(4 rows)



- **Mutual follows** (friends): a pair where each follows the other. Self-join the link table with the columns swapped.
- **Who to follow**: people followed by the people I follow, that I do not follow yet, ranked by how many of my follows follow them.

In [46]:
%%sql
SELECT a.handle AS user_a, b.handle AS user_b
FROM soc_follows f1 JOIN soc_follows f2 ON f1.follower_id = f2.followee_id AND f1.followee_id = f2.follower_id
JOIN soc_users a ON a.user_id = f1.follower_id JOIN soc_users b ON b.user_id = f1.followee_id
WHERE f1.follower_id < f1.followee_id ORDER BY a.handle, b.handle;

SELECT u.handle AS suggestion, count(*) AS followed_by_my_follows
FROM soc_follows mine JOIN soc_follows fof ON fof.follower_id = mine.followee_id
JOIN soc_users u ON u.user_id = fof.followee_id
WHERE mine.follower_id = 1 AND fof.followee_id <> 1
  AND NOT EXISTS (SELECT 1 FROM soc_follows x WHERE x.follower_id = 1 AND x.followee_id = fof.followee_id)
GROUP BY u.user_id ORDER BY followed_by_my_follows DESC, u.handle;

user_a | user_b
-------+-------
ana    | bo    
(1 row)

suggestion | followed_by_my_follows
-----------+-----------------------
di         | 1                     
(1 row)



- **Trending**: top posts by likes using the stored `like_count` (no join to likes needed). A time window would filter on `created_at`.

In [47]:
%%sql
SELECT p.post_id, u.handle, p.body, p.like_count, p.reply_count
FROM soc_posts p JOIN soc_users u ON u.user_id = p.author_id
WHERE p.parent_post_id IS NULL ORDER BY p.like_count DESC, p.post_id LIMIT 3;

post_id | handle | body                           | like_count | reply_count
--------+--------+--------------------------------+------------+------------
1       | ana    | Hello world                    | 3          | 2          
2       | bo     | SQL tip: use keyset pagination | 2          | 0          
6       | di     | New blog post is up            | 2          | 0          
(3 rows)



## 21. Case 3: direct-message inbox with unread counts
- Each member stores `last_read_message_id`. Unread = messages in the thread with a bigger id, sent by someone else.
- The inbox lists each of my threads, the latest message, and the unread count, newest conversation first.
- Ana (user 1) is in threads 1 and 2.

In [48]:
%%sql
SELECT m.thread_id,
       (SELECT group_concat(u.handle, ',') FROM (SELECT u2.handle FROM soc_dm_members x JOIN soc_users u2 ON u2.user_id = x.user_id
                                                 WHERE x.thread_id = m.thread_id AND x.user_id <> m.user_id ORDER BY u2.handle) u) AS with_people,
       (SELECT body FROM soc_dm_messages WHERE thread_id = m.thread_id ORDER BY message_id DESC LIMIT 1) AS last_message,
       (SELECT count(*) FROM soc_dm_messages WHERE thread_id = m.thread_id AND message_id > m.last_read_message_id AND sender_id <> m.user_id) AS unread
FROM soc_dm_members m WHERE m.user_id = 1
ORDER BY (SELECT MAX(message_id) FROM soc_dm_messages WHERE thread_id = m.thread_id) DESC;

thread_id | with_people | last_message | unread
----------+-------------+--------------+-------
2         | cy,di       | Me too       | 1     
1         | bo          | See you      | 1     
(2 rows)



- Mark a thread as read: move the marker to the newest message id. Then unread becomes 0.

In [49]:
%%sql
UPDATE soc_dm_members SET last_read_message_id = (SELECT MAX(message_id) FROM soc_dm_messages WHERE thread_id = 2)
WHERE thread_id = 2 AND user_id = 3;
SELECT m.user_id, m.thread_id,
       (SELECT count(*) FROM soc_dm_messages WHERE thread_id = m.thread_id AND message_id > m.last_read_message_id AND sender_id <> m.user_id) AS unread
FROM soc_dm_members m WHERE m.thread_id = 2 ORDER BY m.user_id;

user_id | thread_id | unread
--------+-----------+-------
1       | 2         | 1     
3       | 2         | 0     
4       | 2         | 0     
(3 rows)



## 22. Case 3: design review, when to denormalize, and 100x scale
- **Denormalized on purpose**: user counters, post counters, the feed table. Each exists because a read is far more frequent than a write, and each has a recount or rebuild query. Rule: normalize first, add a copy only for a measured slow read, and write the check query in the same change.
- **Good**: composite primary keys stop duplicate follows and likes, `CHECK` stops self-follows, `pair_key` makes 1:1 chats unique, keyset pagination, thread root id for single-lookup threads.
- **Weak**: hot rows. Every like of a viral post updates the same `like_count` row, so writers queue up. Deleting a post leaves counters and feed rows to clean. Soft delete and blocking/privacy are not modelled.
- **Missing on purpose**: media, hashtags, mentions, notifications, blocking, private accounts, edit history, message attachments and read receipts per message.
- **At 100x scale I would**:
- Shard by user id. Keep the follow graph in a store built for adjacency lookups, or keep two tables (by follower, by followee) that are both written.
- Keep counters out of the hot row: count events in an append-only table or a cache, and roll them up in batches. Accept slightly stale counts.
- Use hybrid fan-out for feeds (push for normal accounts, pull for accounts with millions of followers) and cap each feed table to the newest N items per user.
- Move search to a search engine (see 08-SQLite-Specials/02-Full-Text-Search-FTS5 for the SQLite version), and paginate everything by keyset.
- Store messages per thread in a partition keyed by `thread_id`, with a separate `unread_count` per member updated on send instead of counted on read.
- Archive old posts to cold storage and keep only indexes needed for recent reads.
- Add soft delete and a moderation audit trail (see 06-Data-Modification/04-History-Audit-And-Temporal-Data).